#### Compare model metrics

In [0]:
from pyspark.sql import functions as SQL_FUNCTIONS

METRICS_TABLE = "workspace.bda_taxi.model_metrics_comparison"

metrics_df = spark.table(METRICS_TABLE)

print("=== Model Performance Comparison ===")
display(
    metrics_df.orderBy(SQL_FUNCTIONS.desc("auc_roc"))
)

# Identify best model by AUC
best_model_row = (
    metrics_df
    .orderBy(SQL_FUNCTIONS.desc("auc_roc"))
    .limit(1)
    .collect()[0]
)

best_model_name = best_model_row["model"]
print(f"Best model based on AUC: {best_model_name}")

#### Compare model distributions

In [0]:
LOGREG_PRED_TABLE = "workspace.bda_taxi.model_preds_logreg"
RF_PRED_TABLE = "workspace.bda_taxi.model_preds_rf"

logreg_preds = spark.table(LOGREG_PRED_TABLE)
rf_preds = spark.table(RF_PRED_TABLE)

preds_all = logreg_preds.unionByName(rf_preds)

print("=== Row counts per model ===")
preds_all.groupBy("model_name").count().show()

#### Compare Confusion Matrix Values (Aggregated)

In [0]:
print("=== Confusion Matrix Summary (Both Models) ===")

confusion_summary = (
    preds_all
    .groupBy("model_name", "label", "prediction")
    .count()
    .orderBy("model_name", "label", "prediction")
)

display(confusion_summary)

#### ROC Curve Comparison

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

def plot_roc_curve(predictions_table: str, model_name: str):
    df = spark.table(predictions_table)
    
    pdf = (
        df.select("label", "probability")
          .toPandas()
    )

    # Extract probability of positive class (index 1)
    pdf["prob_pos"] = pdf["probability"].apply(lambda v: float(v[1]))

    fpr, tpr, _ = roc_curve(pdf["label"], pdf["prob_pos"])
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {roc_auc:.3f})")

plt.figure(figsize=(7, 6))

plot_roc_curve(LOGREG_PRED_TABLE, "Logistic Regression")
plot_roc_curve(RF_PRED_TABLE, "Random Forest")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

#### Select and Refit Best Model

In [0]:
BEST_MODEL_OUTPUT = "workspace.bda_taxi.model_preds_best"

if best_model_name == "LogisticRegression":
    best_preds = spark.table(LOGREG_PRED_TABLE)
else:
    best_preds = spark.table(RF_PRED_TABLE)

(
    best_preds
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(BEST_MODEL_OUTPUT)
)

print(f"Saved best model predictions to {BEST_MODEL_OUTPUT}")